# Quantum Kernel Example Selection for LLM Few Shot Prompting (version 5)
**Run this notebook top to bottom, one cell at a time** (click a cell, press Shift + Enter).
Read the text above each cell before you run it. The Hands-On Guide explains every step in more detail.

* Cells marked **FREE** cost nothing. Cells marked **PAID** call the LLM API.
* Every LLM answer is saved in a cache, so running a cell twice never pays twice.
* If Colab disconnects, run Steps 0 and 1 again and continue where you stopped. Your files live in Google Drive.

**What runs where.**
* 16 selectors (S1 to S8, S10 to S17) share the same pool, prompt and LLM. S9 (1024 shots) is a robustness check.
  S16 and S17 are a quantum and a classical kernel whose input scales are trained (the "best case" for each side).
* 4 datasets: TREC (main), Emotion, DBpedia 14, and Banking77 (77 intents, the harder task; open LLM).
* More qubits with the LLM: 6, 8, 10 and 12 qubits (NCA features) with the open LLM on all four datasets.
* Larger test sets (2,000 items) for Emotion, DBpedia 14 and Banking77 with the open LLM, for statistical power.
* 3 LLMs: the paid main LLM (Claude Haiku 4.5), Qwen2.5 1.5B (free, GPU, 3 seeds everywhere),
  Qwen2.5 7B in 4 bits (free, GPU, one size).
* 2 encoders: MiniLM (main) and MPNet (robustness, TREC, open LLM).
* Studies without the LLM: ablation, qubit scaling (2 to 12 qubits), gate noise. With the open LLM: bandwidth study, k sweep, order.

**Order of priority if time runs out:** Steps 0 to 10a (MUST), 11a and 11b (MUST, free), then the rest.

## Step 0. Put the project in Google Drive (FREE)
1. Upload `qkernel_icl_starter.zip` to the top level of your Google Drive ("My Drive").
2. Turn on the GPU: Runtime > Change runtime type > T4 GPU. (Embeddings and the open LLMs need it.)
3. Run the cell below. It connects Drive, unzips the project once, and moves into the project folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/qkernel_icl_starter'
if not os.path.exists(PROJECT):
    !unzip -q /content/drive/MyDrive/qkernel_icl_starter.zip -d /content/drive/MyDrive/
%cd {PROJECT}
!ls

## Step 1. Install the packages (FREE)
Takes 2 to 4 minutes. Run it again every time Colab starts a new session.

In [ ]:
!pip install -q -r requirements.txt

## Step 2. Check that everything works (FREE)
Every line must start with `PASS`. The test **pennylane matches numpy** is important:
it proves our quantum circuits are the ones we describe in the paper. If it fails, stop and tell the team.

In [ ]:
!python -m src.pipeline check

## Step 3. Run the whole pipeline on fake data (FREE)
Fake texts and fake LLMs, every stage, every study, all three LLM slots and both encoders. Its numbers mean nothing.
It only proves every step runs. It must end with `SMOKE TEST PASSED` (about 20 to 30 minutes).

In [ ]:
!python tests/smoke_pipeline.py

## Step 4. Real data and features (FREE)
Downloads each dataset, turns every text into numbers with all-MiniLM-L6-v2, then shrinks them with
LDA (2 to 5 numbers) and NCA (6 and 8 numbers), fitted on the pool only. The last line repeats this for TREC
with the second encoder (all-mpnet-base-v2, saved in `data/trec__mpnet`). About 15 to 40 minutes in total.
If a download fails, run the cell again.

In [ ]:
DATASETS = ['trec', 'emotion', 'dbpedia']          # the three main datasets
ALL4 = DATASETS + ['banking77']                    # plus the harder task
BIG = ['emotion', 'dbpedia', 'banking77']          # datasets with a 2,000-item test set (TREC has only 500)
for ds in ALL4:
    !python -m src.pipeline features --dataset {ds}
!python -m src.pipeline features --dataset trec --emb mpnet
for ds in BIG:
    !python -m src.pipeline features --dataset {ds} --big

## Step 5. Analyses before the LLM (FREE)
For every dataset, seed (0, 1, 2) and size this tunes the quantum bandwidth on the validation set, runs all
selectors without the LLM and measures overlap (Jaccard), rank agreement, geometric difference,
kernel-target alignment, neighbour vote accuracy and label precision. PennyLane simulates every circuit,
so this takes a while (about 1 to 2 hours). Leave it running.

In [ ]:
for ds in DATASETS:
    !python -m src.pipeline pre_llm --dataset {ds}                       # 3 seeds, LDA 2 to 5
!python -m src.pipeline pre_llm --dataset banking77 --seeds 0
!python -m src.pipeline pre_llm --dataset trec --emb mpnet --seeds 0
for ds in BIG:
    !python -m src.pipeline pre_llm --dataset {ds} --big --seeds 0
for ds in ALL4:
    !python -m src.pipeline pre_llm --dataset {ds} --method nca --seeds 0   # NCA 6 to 12 qubits

### Read the results of Step 5
* `rank_S7_S5` close to 1: the quantum kernel ORDERS the pool like RBF, so the picks will be almost the same.
* `jacc_S5_S7` above 0.9: the quantum kernel picks almost the same examples as RBF.
* `g_vs_rbf` near 1: on this data RBF can do whatever the quantum kernel does.
* `prec_*`: share of picked examples that carry the query's true label (no LLM needed).

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob('results/*/pre_llm_seed*_lda*.json')):
    d = json.load(open(f)); j = d.get('jaccard', {}); p = d.get('label_precision', {}); r = d.get('rank_agreement', {})
    g = lambda k: round(j.get(k, float('nan')), 2)
    rows.append({'dataset': f.split('/')[1], 'setting': f.split('pre_llm_')[1][:-5], 'bandwidth': d['bandwidth'],
                 'g_vs_rbf': round(d['g_quantum_vs_rbf'], 2), 'rank_S7_S5': round(r.get('S7_q_iqp|S5_rbf_lda', float('nan')), 3),
                 'jacc_S5_S6': g('S5_rbf_lda|S6_q_angle'), 'jacc_S5_S7': g('S5_rbf_lda|S7_q_iqp'),
                 'jacc_S7_S9': g('S7_q_iqp|S9_q_iqp_shots'),
                 'prec_S5': round(p.get('S5_rbf_lda', float('nan')), 3), 'prec_S7': round(p.get('S7_q_iqp', float('nan')), 3),
                 'prec_S2': round(p.get('S2_cosine_full', float('nan')), 3), 'prec_S13': round(p.get('S13_nca64_full', float('nan')), 3)})
summary = pd.DataFrame(rows); summary.to_csv('results/summary_pre_llm.csv', index=False); summary

## Step 5b. Studies without the LLM (FREE)
* **Ablation**: twelve variants of the quantum selector, all datasets (about 30 minutes).
* **Qubit scaling**: 2 to 12 qubits with NCA features, all datasets (about 30 to 60 minutes).
* **Gate noise**: density matrices with local depolarizing noise, 5 qubits, all datasets (about 20 minutes each).
  The printed check `global depolarizing ... 1.0` must be exactly 1.0.
You can run this cell later (for example overnight); nothing else depends on it.

In [ ]:
for ds in DATASETS:
    !python -m src.pipeline ablate --dataset {ds}
    !python -m src.pipeline scaling --dataset {ds}
    !python -m src.pipeline noise --dataset {ds}
!python -m src.pipeline scaling --dataset banking77

## Step 6. Connect the main LLM (FREE to set up)
1. Go to console.anthropic.com, add a little credit, and **set a monthly spending limit** under Billing.
2. Create an API key (API Keys > Create Key) and copy it.
3. The model is already chosen: Claude Haiku 4.5, exact id `claude-haiku-4-5-20251001`. Do not change it after the pilot.
4. Run the cell and paste the key into the hidden box. It is never saved in a file.

In [ ]:
import yaml, os, getpass
PROVIDER = 'anthropic'          # or 'openai'
MODEL = 'claude-haiku-4-5-20251001'      # Claude Haiku 4.5

cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['llm']['provider'], cfg['llm']['model'] = PROVIDER, MODEL
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'), sort_keys=False)
key_name = 'ANTHROPIC_API_KEY' if PROVIDER == 'anthropic' else 'OPENAI_API_KEY'
os.environ[key_name] = getpass.getpass(f'Paste your {key_name}: ')
print('Saved provider and model in configs/config.yaml:', PROVIDER, MODEL)

## Step 7. How much will it cost? (FREE)
Counts the different prompts each main run needs (repeats are free thanks to the cache) and estimates tokens.
Multiply by the price per million tokens on the provider's pricing page. The Project Lead approves the budget.

In [ ]:
for ds in DATASETS:
    !python -m src.pipeline estimate --dataset {ds}

## Step 8. Zero shot baseline and pilot on TREC (PAID, small)
* Zero shot: 500 calls with no examples. This is the floor.
* Pilot: all selectors on 100 questions. Read the printed prompt and answer below and check they look right.

In [ ]:
!python -m src.pipeline zeroshot --dataset trec

In [ ]:
!python -m src.pipeline pilot --dataset trec

In [ ]:
import glob, pandas as pd
from src.llm import build_prompt
from src.datasets_registry import spec
pilot = pd.read_csv(sorted(glob.glob('results/trec/runs_llm/seed0_lda*_k4_*_n100.csv'))[0])
train, test = pd.read_csv('data/trec/train.csv'), pd.read_csv('data/trec/test.csv')
pool_idx = pd.read_csv('data/trec/splits_seed0.csv').pool_idx.dropna().astype(int).values
r = pilot[pilot.selector == 'S7_q_iqp'].iloc[0]
ids = list(map(int, r.picked_ids.split()))
print(build_prompt(test.text[r.query_id], train.text.values[pool_idx][ids], train.label.values[pool_idx][ids], spec('trec')))
print('\nLLM said:', repr(r.raw_answer), '| parsed:', r.pred, '| true label:', r.gold)
print('\nInvalid answers in pilot:', round((pilot.pred == 'invalid').mean() * 100, 1), '%  (should be below 5%)')
pilot.groupby('selector')[['correct', 'vote_correct', 'label_prec']].mean().round(3)

## Step 9. Freeze the decision rules (FREE)
Writes `PREREGISTRATION.md` with the rules, the comparison families, the equivalence margin, the tuned values,
the three models and today's date. **Run it once, before any main run, then commit it to GitHub and never edit it.**

In [ ]:
import yaml, json, datetime, os, glob
assert not os.path.exists('PREREGISTRATION.md'), 'Already frozen. Do not overwrite.'
cfg = yaml.safe_load(open('configs/config.yaml')); dec = cfg['decision']; ex = cfg['experiment']
tuned = {f.split('/')[1]: json.load(open(f)) for f in sorted(glob.glob('results/*/tuned.json'))}
fam = lambda name: '; '.join(f'{a} vs {b}' for a, b in dec[name])
text = f'''# Preregistration (frozen {datetime.date.today()})

Main LLM: {cfg["llm"]["provider"]} / {cfg["llm"]["model"]}, temperature {cfg["llm"]["temperature"]}
Second LLM: {cfg["llm2"]["provider"]} / {cfg["llm2"]["model"]}, greedy decoding
Third LLM: {cfg["llm3"]["provider"]} / {cfg["llm3"]["model"]}, 4-bit, greedy, dims {ex["llm3_dims"]}
Encoders: main {cfg["data"]["main_encoder"]}; robustness mpnet (TREC, second LLM)
Selectors: {", ".join(ex["selectors"])}. S14 (label oracle) is an upper bound and is never tested.
S16/S17: one input scale per feature, trained to maximise centred kernel-target alignment on 250 pool items.
Datasets: TREC, Emotion, DBpedia 14 (main); Banking77 (exploratory, open LLM seed 0).
More qubits with the LLM: NCA {cfg["features"]["nca_dims"]} with the open LLM (main LLM: {ex["nca_main_dims"]}), exploratory.
Larger test sets: {cfg["data"]["big_test_size"]} items for Emotion, DBpedia 14, Banking77 (open LLM), exploratory.
Fixed settings: k = {ex["k_main"]}, k sweep {ex["k_sweep"]}, MMR lambda {ex["mmr_lambda"]}, NCA-64 dims {ex["nca_full_dims"]},
LDA dims {cfg["features"]["lda_dims"]}, seeds {cfg["data"]["seeds"]} (main LLM: 3 seeds on TREC, seed 0 elsewhere;
second LLM: 3 seeds on every dataset).

Comparison families (Holm correction within each family, per dataset and LLM):
  Primary:     {fam("primary")}
  Secondary:   {fam("secondary")}
  Exploratory: {fam("exploratory")}
Robustness (TREC, seed 0): S9 (IQP with {cfg["quantum"]["shots"]} shots) vs S7 (exact).
Pooled analysis: cluster bootstrap over queries; TOST equivalence with margin +-{dec["equivalence_margin"]} points
(alpha 0.05, 90% interval inside the margin); minimum detectable effect at 80% power; difficulty terciles.

We say "the quantum kernel did better" only if ALL hold:
1. 95% bootstrap interval of the S7 minus S5 accuracy gap is above 0 and Holm corrected McNemar p < {dec["alpha"]}.
2. The gap is at least {dec["min_gap_points"]} accuracy points.
3. 1 and 2 hold in at least {dec["min_seeds_agree"]} of the seeds run (1 of 1 for a seed 0 only run),
   in at least {dec["min_dims_agree"]} of 4 dims.
4. S7 beats random (S1), and LLM accuracy with S7 beats the neighbour vote with S7.
5. Jaccard(S5, S7) below 0.9 and geometric difference (S7 vs RBF) above 1 in every setting.

No practical difference: in enough settings the whole interval lies between -{dec["min_gap_points"]} and +{dec["min_gap_points"]} points.
Quantum worse: in enough settings the interval lies below 0 and the gap is at least {dec["min_gap_points"]} points.
Mixed: anything else. A claim across datasets needs the same verdict on at least 2 of 3 datasets.

Tuned values (chosen on validation data only): {json.dumps(tuned)}
'''
open('PREREGISTRATION.md', 'w').write(text); print(text)

## Step 10. Main experiments with the main LLM (PAID)
Run the cells in this order. If money or time runs out, stop after any cell: the earlier ones matter most.
If a cell stops halfway (disconnect, rate limit), just run it again. The cache returns every answer already paid for.

In [ ]:
!python -m src.pipeline main --dataset trec --seeds 0      # 10a: TREC seed 0, all dims        (MUST)
!python -m src.pipeline main --dataset emotion --seeds 0   # 10a: second dataset               (MUST)
!python -m src.pipeline main --dataset dbpedia --seeds 0   # 10a: third dataset                (MUST)

In [ ]:
!python -m src.pipeline robust --dataset trec            # 10b: 1024 shots (S9), cheap          (SHOULD)

In [ ]:
!python -m src.pipeline main --dataset trec --seeds 1,2  # 10c: seeds 1 and 2 on TREC           (SHOULD)

In [ ]:
!python -m src.pipeline ksweep --dataset trec            # 10d: k = 2, 8, 16 with the main LLM   (COULD)

In [ ]:
!python -m src.pipeline main --dataset banking77 --seeds 0  # 10f: Banking77 with the main LLM (COULD; long prompts)

In [ ]:
!python -m src.pipeline pre_llm --dataset trec --method nca --seeds 0   # FREE: tuning for NCA
!python -m src.pipeline nca --dataset trec                              # 10e: 6 and 8 qubits (COULD)

## Step 11a. The open LLM, three seeds on every dataset (FREE, GPU)
Qwen2.5 1.5B Instruct answers the **same prompts** (same picks, same order), so any change comes from the LLM alone.
It also stores the probability of every label (seed 0), which gives AUROC and calibration.
First use downloads about 3 GB. Several hours in total: run one dataset per Colab session if needed.

In [ ]:
import torch; print('GPU ready:', torch.cuda.is_available())
for ds in DATASETS:
    !python -m src.pipeline main --dataset {ds} --llm llm2                 # 3 seeds
!python -m src.pipeline main --dataset banking77 --llm llm2 --seeds 0

## Step 11a2. More qubits with the open LLM (FREE, GPU) — MUST
The answer to the reviewers' main question: does the quantum kernel help when it gets 6 to 12 qubits?
Runs NCA features with 6, 8, 10 and 12 qubits on all four datasets (seed 0). The 12-qubit quantum kernels
take a few minutes each to simulate.

In [ ]:
DATASETS = ['trec', 'emotion', 'dbpedia']; ALL4 = DATASETS + ['banking77']
for ds in ALL4:
    !python -m src.pipeline nca --dataset {ds} --llm llm2

## Step 11a3. Larger test sets with the open LLM (FREE, GPU) — SHOULD
2,000 test items instead of 500 for Emotion, DBpedia 14 and Banking77, so the equivalence test has enough power.

In [ ]:
for ds in ['emotion', 'dbpedia', 'banking77']:
    !python -m src.pipeline main --dataset {ds} --big --llm llm2 --seeds 0

## Step 11b. k sweep, order and bandwidth study with the open LLM (FREE, GPU)
* k sweep: k = 2, 8, 16 (k = 4 is the main run) on every dataset.
* Order: the same demonstrations printed nearest last, nearest first and shuffled (TREC).
* Bandwidth study: the quantum selectors at EVERY bandwidth, on validation and test (TREC; add the other
  datasets if time allows). This answers "would tuning with the LLM change the result?".

In [ ]:
for ds in DATASETS:
    !python -m src.pipeline ksweep --dataset {ds} --llm llm2
!python -m src.pipeline order --dataset trec --llm llm2
!python -m src.pipeline bwstudy --dataset trec --llm llm2

## Step 11c. Second encoder (FREE, GPU)
The same pipeline with all-mpnet-base-v2 embeddings on TREC, open LLM, seed 0.

In [ ]:
!python -m src.pipeline main --dataset trec --emb mpnet --llm llm2 --seeds 0

## Step 11d. The larger open LLM, Qwen2.5 7B in 4 bits (FREE, GPU)
Uses about 5.5 GB of GPU memory, so it fits on the T4. First use downloads about 15 GB to the Colab disk.
It runs at one size (d = 5) with the most important selectors. Restart the runtime first
(Runtime > Restart session, then Steps 0 and 1) so the 1.5B model is not still in memory.

In [ ]:
DATASETS = ['trec', 'emotion', 'dbpedia']
for ds in DATASETS:
    !python -m src.pipeline main --dataset {ds} --llm llm3

## Step 11e. OPTIONAL: a few circuits on a real IBM quantum computer
Not needed for the paper. Needs a free IBM Quantum account and its API key (never put the key in a file you share).
First test locally with `--simulate`. Skip this step if you are short of time.

In [ ]:
# !pip install -q qiskit qiskit-ibm-runtime qiskit-aer
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(channel='ibm_quantum_platform', token=getpass.getpass('IBM key: '))
# !python -m src.pipeline hardware --dataset trec --simulate     # local test
# !python -m src.pipeline hardware --dataset trec                # real device

## Step 12. Statistics, verdicts and figures (FREE)
For every dataset, LLM and encoder that has results: comparison tables, **FINAL VERDICT** from the frozen rules,
pooled gaps with the equivalence test, difficulty strata, headroom, per class accuracy and disagreement examples.
Then one summary table and figure across everything.

In [ ]:
import glob
DATASETS = ['trec', 'emotion', 'dbpedia']; ALL4 = DATASETS + ['banking77']
for ds in ALL4:
    for llm in ['llm', 'llm2', 'llm3']:
        if glob.glob(f'results/{ds}/runs_{llm}/seed*_lda*_k4_bw*.csv'):
            !python -m src.pipeline analyze --dataset {ds} --llm {llm}
if glob.glob('results/trec__mpnet/runs_llm2/seed*_k4_bw*.csv'):
    !python -m src.pipeline analyze --dataset trec --emb mpnet --llm llm2
for ds in ['emotion', 'dbpedia', 'banking77']:
    if glob.glob(f'results/{ds}__big/runs_llm2/seed*_k4_bw*.csv'):
        !python -m src.pipeline analyze --dataset {ds} --big --llm llm2
!python -m src.pipeline summary

In [ ]:
import pandas as pd, json
from IPython.display import display
display(pd.read_csv('results/summary_verdicts.csv'))
v = json.load(open('results/trec/runs_llm/verdict.json'))
display(pd.DataFrame(v['pooled']).round(3))
print('Headroom (TREC, main LLM):', v['headroom'])
print('Predict before you pay (TREC):', v['predict_before_pay'])
print('Robustness to shots (TREC):', v.get('robustness_shots'))
display(pd.DataFrame(v['difficulty']).round(3))

## Step 12b. Fill the paper (FREE)
First open `configs/config.yaml` and fill the `paper:` section: the template (`venue`: ieee, springer, acm or arxiv),
author names, affiliation, emails and the GitHub link. Then this cell writes every table, every number, every result
sentence, the author block and every figure, and builds `paper/Paper.docx`. Upload the `paper` folder to Overleaf
and compile `main.tex` (and `supplement.tex`). **Read every sentence once: you are the authors.**

In [ ]:
!python -m src.pipeline paper
!grep -c XYZ paper/results_macros.tex || echo "no XYZ left"
!grep -c fillin paper/results_text.tex || echo "every result sentence written"
!which pandoc > /dev/null || apt-get -qq install -y pandoc > /dev/null
!cd paper && python word/build_docx.py
!cd /content/drive/MyDrive && zip -qr paper_final.zip qkernel_icl_starter/paper -x "*/word/*" && echo "paper_final.zip and paper/Paper.docx are in your Drive"

## Step 13. Back up everything (FREE)
One zip with results, figures, the cache and the preregistration in your Drive. Share it with the team.

In [ ]:
!zip -qr /content/drive/MyDrive/qkernel_results_backup.zip results figures paper PREREGISTRATION.md configs && echo saved